In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hypertension.csv")

print("Shape:", df.shape)
df.head()

Shape: (4240, 16)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [2]:
import numpy as np

print("Column data types:")
print(df.dtypes)

print("\nTrue missing values (NaN) per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget (prevalentHyp) class distribution:")
print(df["prevalentHyp"].value_counts())
print(df["prevalentHyp"].value_counts(normalize=True) * 100)

Column data types:
male                 int64
age                  int64
education          float64
currentSmoker        int64
cigsPerDay         float64
BPMeds             float64
prevalentStroke      int64
prevalentHyp         int64
diabetes             int64
totChol            float64
sysBP              float64
diaBP              float64
BMI                float64
heartRate          float64
glucose            float64
TenYearCHD           int64
dtype: object

True missing values (NaN) per column:
male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

Duplicate rows: 0

Target (prevalentHyp) class distribution:
prevalentHyp
0    2923
1    1317
Name: count, dtype: 

In [3]:
df_clean = df.copy()

# Separate numeric columns by appropriate imputation strategy:
# - Continuous measurements: median
# - Categorical/binary-like: mode (most frequent value)

median_cols = ["cigsPerDay", "totChol", "BMI", "heartRate", "glucose"]
mode_cols = ["education", "BPMeds"]

for col in median_cols:
    median_val = df_clean[col].median()
    missing_count = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"{col}: filled {missing_count} missing values with median = {median_val}")

for col in mode_cols:
    mode_val = df_clean[col].mode()[0]
    missing_count = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"{col}: filled {missing_count} missing values with mode = {mode_val}")

print("\nMissing values AFTER imputation (should all be 0):")
print(df_clean.isnull().sum())

# Save cleaned dataset
df_clean.to_csv("../data/processed/hypertension_clean.csv", index=False)
print("\nCleaned dataset saved to ml/data/processed/hypertension_clean.csv")

cigsPerDay: filled 29 missing values with median = 0.0
totChol: filled 50 missing values with median = 234.0
BMI: filled 19 missing values with median = 25.4
heartRate: filled 1 missing values with median = 75.0
glucose: filled 388 missing values with median = 78.0
education: filled 105 missing values with mode = 1.0
BPMeds: filled 53 missing values with mode = 0.0

Missing values AFTER imputation (should all be 0):
male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64

Cleaned dataset saved to ml/data/processed/hypertension_clean.csv


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Separate features and target
X = df_clean.drop(["prevalentHyp", "TenYearCHD"], axis=1)
y = df_clean["prevalentHyp"]

# 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Compare all 4 required algorithms
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42, probability=True)
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring="accuracy")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("CV Mean Accuracy", ascending=False)
results_df

Training set: (3392, 14)
Testing set: (848, 14)


D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in ve

,Model,CV Mean Accuracy,CV Std,Test Accuracy,Test Precision,Test Recall,Test F1
0,Random Forest,0.900056,0.013564,0.887972,0.806569,0.840304,0.823091
3,Support Vector Machine,0.882372,0.014814,0.876179,0.813492,0.779468,0.796117
2,K-Nearest Neighbors,0.857318,0.012579,0.860849,0.822222,0.703422,0.758197
1,Decision Tree,0.847298,0.019458,0.853774,0.762264,0.768061,0.765152


In [5]:
import joblib

best_model = models["Random Forest"]

# What does the model rely on most?
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importances:")
print(importances.round(3))

# Exact feature order the Flask API must use
print("\nFeature order (copy this into Flask):")
print(X.columns.tolist())

# Save model and scaler
joblib.dump(best_model, "../saved_models/hypertension_model.pkl")
joblib.dump(scaler, "../saved_models/hypertension_scaler.pkl")
print("\nHypertension model and scaler saved successfully.")

Feature importances:
sysBP              0.399
diaBP              0.237
BMI                0.072
age                0.066
totChol            0.053
heartRate          0.046
glucose            0.045
BPMeds             0.027
cigsPerDay         0.021
education          0.016
male               0.008
currentSmoker      0.006
diabetes           0.002
prevalentStroke    0.001
dtype: float64

Feature order (copy this into Flask):
['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

Hypertension model and scaler saved successfully.
